# TetraFT — QAFT for 2-bit Quaternary LLMs

Quantization-Aware Fine-Tuning on **Qwen2.5-0.5B** using the quaternary grid
{-1, -c, c, 1} with Straight-Through Estimator.

Runs on a single T4 (free Colab) in under 2 hours.

In [ ]:
# @title 1. Clone TetraFT from GitHub
import sys, os, json, math, logging

!git clone https://github.com/falloficaruss/tetraft.git /content/tetraft
sys.path.insert(0, '/content/tetraft')

!pip install datasets --upgrade -q
print('TetraFT cloned from GitHub')

In [ ]:
# @title 2. Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from config import QAFTConfig
from model import replace_linear_layers
from train import QAFTTrainer
from eval import evaluate_perplexity
from quantize import QuantizedLinear

logging.basicConfig(level=logging.INFO, format='%(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"})')

In [ ]:
# @title 3. Configuration
cfg = QAFTConfig(
    model_name='Qwen/Qwen2.5-0.5B',
    quaternary_c=0.25,
    learning_rate=2e-4,
    batch_size=2,
    seq_length=512,
    max_steps=5000,
    warmup_steps=500,
    gradient_accumulation_steps=4,
    logging_steps=50,
    eval_steps=500,
    save_steps=1000,
    gradient_checkpointing=True,
    quant_warmup=True,
    output_dir='./checkpoints',
)
print(cfg)

In [ ]:
# @title 3b. Mount Drive & set checkpoint path (persists across sessions)
from google.colab import drive
drive.mount('/content/drive')

cfg.output_dir = '/content/drive/MyDrive/tetraft_checkpoints'
os.makedirs(cfg.output_dir, exist_ok=True)
print(f'Checkpoints will be saved to: {cfg.output_dir}')

In [ ]:
# @title 4. Load Model & Tokenizer
print('Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float32,
)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Model loaded to CPU (will move to GPU after layer replacement)')

In [ ]:
# @title 5. Replace Linear Layers with Quaternary
n_linear = sum(1 for _ in model.named_modules() if isinstance(_[1], nn.Linear))
print(f'Original nn.Linear layers: {n_linear}')

replace_linear_layers(model, c=cfg.quaternary_c, skip_lm_head=cfg.skip_lm_head)

n_quantized = sum(1 for _ in model.named_modules() if isinstance(_[1], QuantizedLinear))
print(f'Replaced with QuantizedLinear: {n_quantized}')
model.to(device)
print(f'Model moved to {device}')

In [ ]:
# @title 6. Prepare Dataset (FineWeb sample)
print('Loading FineWeb sample-10BT (streaming)...')
dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    "sample-10BT",
    split="train",
    streaming=True,
)
sample = []
for i, doc in enumerate(dataset):
    if i >= 10_000:
        break
    sample.append(doc['text'])

def tokenize_fn(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=cfg.seq_length,
        return_tensors='pt',
    )

train_texts = sample[:9_000]
eval_texts = sample[9_000:]

def collate(batch):
    batch = tokenize_fn([b['text'] for b in batch])
    batch['labels'] = batch['input_ids'].clone()
    return batch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return {'text': self.texts[i]}

train_dataset = TextDataset(train_texts)
eval_dataset = TextDataset(eval_texts)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate, num_workers=0)

print(f'Train batches: {len(train_loader)}, Eval batches: {len(eval_loader)}')

In [ ]:
# @title 7. Train
trainer = QAFTTrainer(model, tokenizer, cfg)
trainer.train(train_loader, eval_loader)

In [ ]:
# @title 8. Evaluate Final Perplexity
ppl = evaluate_perplexity(model, eval_loader, max_batches=20)
print(f'Final perplexity: {ppl:.2f}')

In [ ]:
# @title 9. Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(trainer.metrics["step"], trainer.metrics["loss"])
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")

axes[1].plot(trainer.metrics["step"], trainer.metrics["lr"])
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("LR Schedule")

if trainer.metrics["perplexity"]:
    ppl_steps, ppl_vals = zip(*trainer.metrics["perplexity"])
    axes[2].plot(ppl_steps, ppl_vals, marker="o")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Perplexity")
    axes[2].set_title("Eval Perplexity")
    axes[2].axhline(y=trainer.best_perplexity, color="r", linestyle="--", label=f"Best: {trainer.best_perplexity:.2f}")
    axes[2].legend()

plt.tight_layout()
plt.show()

# Checkpoints saved automatically to Drive: tetraft_checkpoints/
# Resume: trainer.load_checkpoint("...checkpoint-best"); trainer.train(train_loader, eval_loader)

## Results

After training completes, record:
- **Pre-QAFT perplexity** (run eval before layer replacement as baseline)
- **Post-QAFT perplexity** (after training)
- **Perplexity delta** = post - pre (should be small, ideally < 2-3)

The quaternary compression saves **~87.5%** of linear layer parameter memory:
FP32 to 2-bit equivalent: 32 bits to 2 bits per weight.